Import libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")
from pycomet.networks import MatExpME

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

# path_to_figs = "./figs/"
# os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
  x=None,
  y=None,
  xref=None,
  yref=None,
  labels=[r'$t$ [s]', r'$X$'],
  scales=['log', 'log'],
  names=None,
  figname=None
):
  # Initialize figures
  fig = plt.figure()
  ax = fig.add_subplot()
  # x axis
  ax.set_xlabel(labels[0])
  ax.set_xscale(scales[0])
  # y axis
  ax.set_ylabel(labels[1])
  ax.set_yscale(scales[1])
  # Plotting
  n, d = y.shape if (x is not None) else yref.shape
  true_style = dict(
    lw=1
  )
  pred_style = dict(
    marker='^',
    fillstyle='none',
    markersize=5,
    markevery=n//20,
    ls='--',
    lw=1
  )
  for i in range(d):
    if (x is not None):
      ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
    if (xref is not None):
      ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
    ax.plot([], [], c=COLORS[i], label=names[i])
  plt.legend()
  if (figname is not None):
    fig.savefig(figname)
  plt.show()
  plt.close()

Load master equation model

In [ ]:
model = MatExpME(
  rates="./datagen/data/physics/rates.hdf5",
  specie_thermos={
    "filename": "./physics/thermo/specie_thermos_ng.json",
    "config": None
  },
  use_corr=False,
  use_state_corr=False
)

Get solution

In [ ]:
nb = np.random.randint(1,10)
path_to_case = f"./datagen/data/test/case_{str(nb).zfill(6)}.csv"

In [ ]:
def get_data(sol):
  t, na, T = [sol[k].to_numpy().reshape(-1,1) for k in ("t", "n_O", "T")]
  nm0 = sol[["n_O2_1_0", "n_O2_2_0", "n_O2_3_0"]].to_numpy()
  nm = sol[["O2_1", "O2_2", "O2_3"]].to_numpy()
  return t, na, T, nm0, nm

In [ ]:
sol = pd.read_csv(path_to_case)
t, na, T, nm0, nm = get_data(sol)

Get predicted solution

In [ ]:
x = {
  "qt": np.hstack([t, na, T]),
  "n": nm0,
  "f": None
}
xl = list(x.values())
nm_pred = model(xl).numpy().squeeze()

In [ ]:
plot_var(
  x=t.squeeze()[1:],
  y=nm_pred[1:],
  xref=t.squeeze()[1:],
  yref=nm[1:],
  labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
  scales=['log', 'linear'],
  names=[r'O$_2^{(%d)}$' % (i+1) for i in range(nm.shape[1])]
)

In [ ]:
plot_var(
  x=t.squeeze()[1:],
  y=nm_pred[1:],
  xref=t.squeeze()[1:],
  yref=nm[1:],
  labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
  scales=['log', 'log'],
  names=[r'O$_2^{(%d)}$' % (i+1) for i in range(nm.shape[1])]
)